# Лабораторная работа 7

**Тема:** решение нелинейных уравнений.

**Выполнил:** Ипатенков Григорий Александрович, ИВТ-4, вариант 6

**Вариант 6:** $\;a x + \sin x - b = 0$

**Задание:**
1. Написать программу поиска корня $f(x) = 0$ методами дихотомии, хорд, простой итерации и Ньютона.
2. Исходные параметры процедур: начальное приближение $x_0$, точность $\varepsilon$, максимальное число итераций $N$.
3. Точность $\varepsilon$ задаётся в разумных пределах, не выше погрешности вычислений.
4. Задать параметры $a, b$ так, чтобы задача имела решение.
5. Построить график функции и графически отделить корень, определив интервал локализации $[A, B]$.
6. Проверить условия сходимости всех методов.
7. Выбрать «хорошее» начальное приближение.
8. Остановка при достижении точности или по исчерпании числа итераций.
9. На выходе — значение корня и число итераций.
10. Протестировать программу на квадратном уравнении с точными корнями через дискриминант.
11. Обратить внимание на способы вычисления логарифмов в Python.

## Импорт и параметры

Параметры уравнения: $a = 0.5$, $b = 1$, то есть

$$f(x) = 0.5x + \sin x - 1, \qquad f'(x) = 0.5 + \cos x, \qquad f''(x) = -\sin x.$$

Точность $\varepsilon = 10^{-6}$. Машинная погрешность `float` около $2.2\cdot10^{-16}$, поэтому
такая точность достижима. Требовать $\varepsilon$ порядка $10^{-16}$ бессмысленно: это
погрешность самих вычислений (пункт 3 задания).

In [ ]:
import math
from itertools import pairwise

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.dpi"] = 100

In [ ]:
ASK_INPUT = False   # True — ввод параметров с клавиатуры при запуске


def ask(prompt, default, cast=float):
    if not ASK_INPUT:
        return default
    s = input(f"{prompt} [{default}]: ").strip()
    return cast(s) if s else default


a = ask("Параметр a", 0.5)
b = ask("Параметр b", 1.0)
EPS = ask("Точность ε", 1e-6)
N_MAX = ask("Максимальное число итераций N", 100, int)

assert EPS >= 10 * np.finfo(float).eps, "ε меньше погрешности вычислений"


def f(x):
    return a * x + math.sin(x) - b


def df(x):
    return a + math.cos(x)


def d2f(x):
    return -math.sin(x)


print(f"f(x) = {a}·x + sin(x) − {b}")
print(f"ε = {EPS},  N = {N_MAX},  машинный эпсилон = {np.finfo(float).eps:.1e}")

## 1. Отделение корня (1 этап)

**Аналитически.** $f'(x) = 0.5 + \cos x$ обращается в ноль при $x = 2\pi/3 \approx 2.09$ и $x = 4\pi/3 \approx 4.19$.
Это максимум и минимум:

* $f(2.09) \approx 0.91 > 0$;
* $f(4.19) \approx 0.23 > 0$.

При $x \to -\infty$ функция уходит в $-\infty$, при $x \to +\infty$ — в $+\infty$. Значит, корень **единственный**,
и он лежит левее $2\pi/3$.

**Графически** удобно записать уравнение как $f_1(x) = f_2(x)$: $\;0.5x - 1 = -\sin x$.
Корень — абсцисса точки пересечения прямой и синусоиды.

In [ ]:
xs = np.linspace(-2, 8, 800)
fx = np.array([f(x) for x in xs])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(xs, fx, "b", lw=2, label="f(x) = 0.5x + sin x − 1")
ax1.axhline(0, color="k", lw=0.8)
ax1.axvspan(0.5, 1.0, color="g", alpha=0.15, label="интервал локализации [0.5; 1]")
ax1.set_title("График f(x)")
ax1.set_xlabel("x")
ax1.set_ylabel("f(x)")
ax1.grid(alpha=0.4)
ax1.legend()

ax2.plot(xs, a * xs - b, "r", lw=2, label="f₁(x) = 0.5x − 1")
ax2.plot(xs, -np.sin(xs), "b", lw=2, label="f₂(x) = −sin x")
ax2.axvspan(0.5, 1.0, color="g", alpha=0.15)
ax2.set_title("Отделение корня: пересечение f₁(x) = f₂(x)")
ax2.set_xlabel("x")
ax2.grid(alpha=0.4)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"{'x':>6}{'f(x)':>12}")
for x in np.arange(0, 1.51, 0.25):
    print(f"{x:>6.2f}{f(x):>12.5f}")

По графику и таблице знак меняется между 0.5 и 1. Берём интервал локализации $[A, B] = [0.5;\ 1]$.

## 2. Проверка условий сходимости

| Метод | Условие |
|---|---|
| Дихотомия | $f$ непрерывна, $f(A)\,f(B) < 0$ |
| Хорды | то же, плюс $f'$ и $f''$ сохраняют знак на $[A, B]$. Неподвижен тот конец, где $f \cdot f'' > 0$ |
| Простая итерация | $\lvert\varphi'(x)\rvert \le q < 1$ на $[A, B]$ и $\varphi([A, B]) \subset [A, B]$ |
| Ньютон | $f'$ и $f''$ не обращаются в ноль и сохраняют знак, $f(x_0)\,f''(x_0) > 0$ |

**Запись $x = \varphi(x)$ для простой итерации** по формуле (6): $\varphi(x) = x + g\,f(x)$, $g = \text{const}$.
Тогда $\varphi'(x) = 1 + g\,f'(x)$. Если $m \le f'(x) \le M$, то при $g = -2/(m + M)$ получается
наименьшее значение $\max|\varphi'|$, равное $(M - m)/(M + m)$.

Для сравнения проверим и «неудачную» запись $x = 2(1 - \sin x)$: у неё $|\varphi'| = 2|\cos x| > 1$ возле корня.

In [ ]:
A, B = 0.5, 1.0
grid = np.linspace(A, B, 10001)
d1 = np.array([df(x) for x in grid])
d2 = np.array([d2f(x) for x in grid])

m1, M1 = d1.min(), d1.max()
G = -2 / (m1 + M1)


def phi(x):                       # удачная запись (6)
    return x + G * f(x)


def dphi(x):
    return 1 + G * df(x)


def phi_bad(x):                   # «неудачная» запись x = 2(1 − sin x)
    return 2 * (1 - math.sin(x))


q = max(abs(dphi(x)) for x in grid)
q_bad = min(abs(-2 * math.cos(x)) for x in grid)
phi_vals = [phi(x) for x in grid]

def ok(cond):
    return "выполнено ✓" if cond else "НЕ выполнено ✗"


print(f"f(A) = {f(A):.5f},  f(B) = {f(B):.5f}")
print(f"Дихотомия:  f(A)·f(B) < 0                     — {ok(f(A) * f(B) < 0)}")
print(f"Хорды:      f' ∈ [{m1:.3f}; {M1:.3f}] > 0,  f'' ∈ [{d2.min():.3f}; {d2.max():.3f}] < 0 — {ok(m1 > 0 and d2.max() < 0)}")
fixed = A if f(A) * d2f(A) > 0 else B
print(f"            неподвижный конец хорды: {fixed} (там f·f'' > 0), x0 = {B if fixed == A else A}")
print(f"Итерация:   g = −2/(m + M) = {G:.4f},  max|φ'| = {q:.4f} < 1           — {ok(q < 1)}")
print(f"            φ([A, B]) = [{min(phi_vals):.4f}; {max(phi_vals):.4f}] ⊂ [A, B]  — {ok(A <= min(phi_vals) and max(phi_vals) <= B)}")
print(f"            для x = 2(1 − sin x): min|φ'| на [A, B] = {q_bad:.3f} > 1 — процесс разойдётся")
print(f"Ньютон:     f'·f'' < 0 (разные знаки) → по правилу x0 = A = {A}")
print(f"            f(x0)·f''(x0) = {f(A) * d2f(A):.4f} > 0             — {ok(f(A) * d2f(A) > 0)}")

## 3. Реализация методов (2 этап — уточнение корня)

Все процедуры получают $\varepsilon$ и $N$ и возвращают корень, число итераций и историю приближений
для графиков.

* **Дихотомия.** Отрезок делится пополам, пока его длина не станет меньше $2\varepsilon$. Корень — середина последнего отрезка.
* **Хорды**, формулы (14)–(15). Один конец $c$ неподвижен (там, где $f\,f'' > 0$), второй движется:
  $x_{n+1} = x_n - \dfrac{f(x_n)(x_n - c)}{f(x_n) - f(c)}$. Остановка по условию (3): $|x_{n+1} - x_n| < \varepsilon$.
* **Простая итерация**, формула (5): $x_{n+1} = \varphi(x_n)$. Остановка по (7).
* **Ньютон**, формула (8): $x_{n+1} = x_n - f(x_n)/f'(x_n)$. Остановка по (9).

In [ ]:
def bisection(f, A, B, eps, n_max):
    if f(A) * f(B) > 0:
        raise ValueError("на концах интервала нет смены знака")
    hist = []
    for n in range(1, n_max + 1):
        c = (A + B) / 2
        hist.append(c)
        if f(c) == 0:
            return c, n, hist
        if f(A) * f(c) < 0:
            B = c
        else:
            A = c
        if B - A < 2 * eps:
            c = (A + B) / 2
            hist.append(c)
            return c, n, hist
    return (A + B) / 2, n_max, hist


def chord(f, A, B, eps, n_max, d2f):
    c, x = (A, B) if f(A) * d2f(A) > 0 else (B, A)   # c — неподвижный конец
    hist = [x]
    for n in range(1, n_max + 1):
        x_new = x - f(x) * (x - c) / (f(x) - f(c))    # (15)
        hist.append(x_new)
        if abs(x_new - x) < eps:                        # (3)
            return x_new, n, hist
        x = x_new
    return x, n_max, hist


def simple_iteration(phi, x0, eps, n_max):
    x = x0
    hist = [x]
    for n in range(1, n_max + 1):
        x_new = phi(x)                                  # (5)
        hist.append(x_new)
        if not math.isfinite(x_new) or abs(x_new) > 1e12:
            return x_new, n, hist                       # расходится
        if abs(x_new - x) < eps:                        # (7)
            return x_new, n, hist
        x = x_new
    return x, n_max, hist


def newton(f, df, x0, eps, n_max):
    x = x0
    hist = [x]
    for n in range(1, n_max + 1):
        d = df(x)
        if d == 0:
            raise ZeroDivisionError("f'(x) = 0, метод Ньютона неприменим")
        x_new = x - f(x) / d                            # (8)
        hist.append(x_new)
        if abs(x_new - x) < eps:                        # (9)
            return x_new, n, hist
        x = x_new
    return x, n_max, hist

## 4. Тестирование на квадратном уравнении (пункт 10)

Тестовый пример: $\;x^2 - x - 6 = 0$.

Аналитическое решение через дискриминант:

$$D = b^2 - 4ac = (-1)^2 - 4 \cdot 1 \cdot (-6) = 25, \qquad x_{1,2} = \frac{1 \pm \sqrt{25}}{2} \;\Rightarrow\; x_1 = 3,\ x_2 = -2.$$

Интервалы локализации: $[2.5;\ 4]$ для $x_1 = 3$ и $[-3;\ -1.5]$ для $x_2 = -2$.
Для простой итерации записи такие:
* для $x_1$: $x = \sqrt{x + 6}$, $\varphi' = \dfrac{1}{2\sqrt{x + 6}} \approx 0.17$;
* для $x_2$: $x = -\sqrt{x + 6}$, $|\varphi'| \approx 0.25$.

In [ ]:
def q_f(x):
    return x ** 2 - x - 6


def q_df(x):
    return 2 * x - 1


def q_d2f(x):
    return 2.0


D = (-1) ** 2 - 4 * 1 * (-6)
roots_exact = [(1 + math.sqrt(D)) / 2, (1 - math.sqrt(D)) / 2]
print(f"x² − x − 6 = 0:  D = {D},  x1 = {roots_exact[0]},  x2 = {roots_exact[1]}\n")

tests = [
    (roots_exact[0], 2.5, 4.0, lambda x: math.sqrt(x + 6), 4.0),
    (roots_exact[1], -3.0, -1.5, lambda x: -math.sqrt(x + 6), -3.0),
]

print(f"{'Корень':>7} {'Метод':<18}{'Найдено':>16}{'Итераций':>10}{'|x − x_точн|':>15}")
print("-" * 68)
for exact, A_t, B_t, phi_t, x0_newton in tests:
    results = [
        ("дихотомия", bisection(q_f, A_t, B_t, EPS, N_MAX)),
        ("хорды", chord(q_f, A_t, B_t, EPS, N_MAX, q_d2f)),
        ("простая итерация", simple_iteration(phi_t, (A_t + B_t) / 2, EPS, N_MAX)),
        ("Ньютон", newton(q_f, q_df, x0_newton, EPS, N_MAX)),
    ]
    for name, (x, n, _) in results:
        print(f"{exact:>7.0f} {name:<18}{x:>16.10f}{n:>10}{abs(x - exact):>15.2e}")
    print()

Все четыре метода находят оба корня с ошибкой меньше $\varepsilon = 10^{-6}$. Значит, программа работает
правильно, и её можно применять к уравнению варианта.

## 5. Решение уравнения варианта 6

In [ ]:
x_bis, n_bis, h_bis = bisection(f, A, B, EPS, N_MAX)
x_chd, n_chd, h_chd = chord(f, A, B, EPS, N_MAX, d2f)
x_it, n_it, h_it = simple_iteration(phi, (A + B) / 2, EPS, N_MAX)
x_nw, n_nw, h_nw = newton(f, df, A, EPS, N_MAX)
x_bad, n_bad, h_bad = simple_iteration(phi_bad, (A + B) / 2, EPS, 20)

# эталон для оценки ошибки — Ньютон с точностью до машинной
x_ref = newton(f, df, A, 1e-15, 100)[0]

rows = [
    ("Дихотомия", x_bis, n_bis),
    ("Хорды", x_chd, n_chd),
    ("Простая итерация", x_it, n_it),
    ("Ньютон", x_nw, n_nw),
]
print(f"Уравнение: {a}x + sin x − {b} = 0,  [A, B] = [{A}, {B}],  ε = {EPS}\n")
print(f"{'Метод':<20}{'Корень':>16}{'Итераций':>10}{'|f(x)|':>12}{'|x − x*|':>12}")
print("-" * 70)
for name, x, n in rows:
    print(f"{name:<20}{x:>16.10f}{n:>10}{abs(f(x)):>12.2e}{abs(x - x_ref):>12.2e}")

print(f"\nЭталон x* = {x_ref:.15f}")
print(f"Неудачная запись x = 2(1 − sin x): за {n_bad} итераций x = {x_bad:.4f} — процесс не сходится")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# сходимость: ошибка по итерациям
ax = axes[0, 0]
for name, h, st in [("Дихотомия", h_bis, "o-"), ("Хорды", h_chd, "s-"),
                    ("Простая итерация", h_it, "^-"), ("Ньютон", h_nw, "d-")]:
    err = [max(abs(x - x_ref), 1e-17) for x in h]
    ax.semilogy(range(len(err)), err, st, ms=4, label=f"{name} ({len(h) - 1} ит.)")
ax.axhline(EPS, color="r", ls="--", label="ε")
ax.set_title("Скорость сходимости: |xₙ − x*|")
ax.set_xlabel("номер итерации n")
ax.grid(alpha=0.4, which="both")
ax.legend()

# Ньютон: касательные
ax = axes[0, 1]
xx = np.linspace(0.4, 1.05, 300)
ax.plot(xx, [f(x) for x in xx], "b", lw=2, label="f(x)")
ax.axhline(0, color="k", lw=0.8)
for xn in h_nw[:3]:
    x1 = xn - f(xn) / df(xn)
    ax.plot([xn, x1], [f(xn), 0], "r-", lw=1)
    ax.plot([xn, xn], [0, f(xn)], "r:", lw=1)
    ax.plot(xn, f(xn), "ro", ms=4)
ax.set_title("Метод Ньютона: касательные из x₀ = A")
ax.set_xlabel("x")
ax.grid(alpha=0.4)
ax.legend()


def cobweb(ax, phi_func, hist, title, lim):
    xx = np.linspace(*lim, 300)
    ax.plot(xx, [phi_func(x) for x in xx], "b", lw=2, label="y = φ(x)")
    ax.plot(xx, xx, "k", lw=1, label="y = x")
    for x0, x1 in pairwise(hist):
        ax.plot([x0, x0], [x0, x1], "r", lw=0.9)
        ax.plot([x0, x1], [x1, x1], "r", lw=0.9)
    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.grid(alpha=0.4)
    ax.legend()


cobweb(axes[1, 0], phi, h_it, f"Простая итерация, φ(x) = x + g·f(x): max|φ'| = {q:.2f}", (0.45, 1.05))
cobweb(axes[1, 1], phi_bad, h_bad[:8], "Неудачная запись x = 2(1 − sin x): |φ'| > 1, расходится", (-0.2, 2.2))

plt.tight_layout()
plt.show()

## 6. Логарифмы в Python (пункт 11)

В математике $\ln$ — натуральный логарифм, $\lg$ — десятичный. В Python имена другие, и их легко перепутать:

| Математика | `math` | `numpy` |
|---|---|---|
| $\ln x$ | `math.log(x)` | `np.log(x)` |
| $\lg x$ | `math.log10(x)` | `np.log10(x)` |
| $\log_2 x$ | `math.log2(x)` | `np.log2(x)` |
| $\log_a x$ | `math.log(x, a)` | `np.log(x) / np.log(a)` |
| $\ln(1 + x)$ при малых $x$ | `math.log1p(x)` | `np.log1p(x)` |

`math.log(x)` — это **натуральный** логарифм, а не десятичный. Для $x \le 0$ функция `math.log`
выбрасывает `ValueError`, а `np.log` возвращает `nan` или `-inf` с предупреждением.

In [ ]:
x = 100.0
print(f"math.log({x})     = {math.log(x):.6f}   (ln)")
print(f"math.log10({x})   = {math.log10(x):.6f}   (lg)")
print(f"math.log2({x})    = {math.log2(x):.6f}   (log₂)")
print(f"math.log({x}, 5)  = {math.log(x, 5):.6f}   (log₅)")
print(f"math.log1p(1e-17) = {math.log1p(1e-17):.1e},  а math.log(1 + 1e-17) = {math.log(1 + 1e-17):.1e} — потеря точности")
try:
    math.log(-1)
except ValueError as e:
    print(f"math.log(-1)      → ValueError: {e}")
with np.errstate(invalid="ignore", divide="ignore"):
    print(f"np.log(-1) = {np.log(-1.0)},  np.log(0) = {np.log(0.0)}")

## Вывод

1. Уравнение $0.5x + \sin x - 1 = 0$ имеет единственный корень $x^* \approx 0.7046$. Он отделён графически
   и таблицей значений на интервале $[0.5;\ 1]$.
2. Условия сходимости всех методов на этом интервале выполнены. $f' > 0$ и $f'' < 0$ (случай «в» на рис. 9),
   поэтому у хорд неподвижен конец $A$, а Ньютон стартует с $x_0 = A$, где $f(x_0) f''(x_0) > 0$.
3. **Дихотомия** гарантированно сходится, но медленнее всех: около 20 итераций, отрезок уменьшается вдвое за шаг.
   **Хорды** и **простая итерация** сходятся линейно и быстрее. **Ньютон** сходится квадратично: ему хватает
   3–4 итераций, число верных знаков примерно удваивается за шаг.
4. Успех метода простой итерации зависит от записи $x = \varphi(x)$. Запись по формуле (6) с $g = -2/(m + M)$
   даёт $|\varphi'| \approx 0.14$ и быструю сходимость. «Лобовая» запись $x = 2(1 - \sin x)$ имеет $|\varphi'| > 1$ и расходится.
5. Тест на квадратном уравнении $x^2 - x - 6 = 0$ (корни 3 и −2) подтвердил правильность всех процедур.